# Win outcomes scout — INVENTORY.md verification notebook

## TL;DR

**What this is.** A scouting pass for an upcoming exploratory analysis on whether Win product activity correlates with electoral outcomes. Win is GoodParty's product for independent candidates running for office. The scout characterizes what data is already available to answer that question. It does not build new joins, run the correlation analysis itself, or recommend a specific model.

**Scope.** Candidacies in the 2025-05 through 2026-12 cycle window. Unit of analysis is the candidacy (one row per cycle, so a candidate running in two cycles appears twice). Outcome is `latest_stage_reached` paired with `latest_stage_result`. Baseline is all registered Win users including those with zero product engagement. ICP eligibility (`icp_office_win`) is treated as a slicing dimension, not a filter.

**This artifact.** `../INVENTORY.md` holds findings, gaps, and recommended actions. This notebook (`inventory_queries.ipynb`) holds the queries that produce every coverage number cited in INVENTORY.md, parameterized so the cycle window and other filters can be adjusted to reproduce or extend.

**Findings.** Four data sources are in scope, all in Databricks: Win product activity (Amplitude events), candidacy and election outcomes (the civics mart), candidate and user metadata (the product database), and the segmentation dimensions that link them (office level, state, party, ICP flags). These join cleanly at user grain. The gaps are inside specific fields, not in the joins. The analytical work going forward is in cutting each question correctly, not in stitching the data together.

- **Available.** Win product, civics, and Amplitude data in Databricks all join cleanly at user grain, with 96%+ Amplitude join coverage in the 2025-05 to 2026-12 cycle window. Stable keys link product activity, candidacy, outcomes, and segmentation. Any question that fits inside this universe is reachable.
- **Gaps.** Coverage drops are inside columns, not between tables. Final outcome labels populate only a sub-segment of candidacies (the Did-You-Win modal adds ~970 net new labels but skews ~80% wins via response selection). Viability is sparse on the BR-only `viability_score` and only partially wired in via the better-covered TS-primary `viability_rating_2_0`. Segmentation dimensions like election level, judicial, and appointed thin out at narrow slices. Each modeled Amplitude event has its own instrumentation start date that further bounds who is eligible for any feature-effect analysis.
- **Implication.** Every correlation question is its own narrow cut, not a single global model. The shape is: pick a feature, pin its launch date and a meaningful exposure window, identify the candidacies whose election date sits after that window, intersect with whichever candidacies actually have an outcome label, then correlate. Recent features have small eligible pools. Old features cover everyone but discriminate poorly. The deliberate shaping is the analysis.

## About this notebook

This notebook produces every numeric claim cited in `../INVENTORY.md`. Cells are organized by `[NB-N]` references that map back to INVENTORY sections — each cell's markdown header states which Source / subsection it produces numbers for.

**Cycle window and other knobs** are defined as Python variables in the Setup cell (`WINDOW_START`, `WINDOW_END`). Changing them and re-running the rest of the notebook regenerates all the numbers under the new scope. The default window is the resolved scope (`2025-05-01` to `2026-12-31`).

**Execution:** Databricks Workspace (preferred) or via Databricks Connect from a local kernel. See `analytics/win_churn/WORKFLOW.md` for the sync pattern.

**Memory reminder:** project memory says for ad-hoc queries, hit `goodparty_data_catalog.*` directly; `ref()` can resolve to stale dev artifacts.

## INVENTORY.md → notebook cross-reference

| INVENTORY section | Cells |
|---|---|
| Completeness summary (top of doc) | NB-1, NB-2, NB-4, NB-6, NB-7a |
| Source 1.2 — `users_win_candidacy` coverage | NB-1, NB-2 |
| Source 1.4 — segmentation dimensions (election_level, is_pro, ICP) | NB-4, NB-6, NB-11 |
| Source 1.5 — outcome variables, `candidacy_result` fallback hazard | NB-3 |
| Source 2.2 — civics-mart cycle slice | NB-5 |
| Source 3.1a — Amplitude universe vs modeled | NB-9a, NB-9b |
| Source 3.5 — Did You Win Modal payload + reconciliation | NB-9c.1, NB-9c.2, NB-9c.3 |
| Source 3.8 — recommended-action funnel feasibility | NB-9b (event families ≥ users threshold) |
| Source 4.3 — viability coverage by source bucket | NB-7a |
| Source 4.5 — score-band distribution | NB-7a (band table) |
| Source 4.7 — viability calibration on closed races | NB-7b (pending run) |
| Source 5.4 — product DB tables for funnel reconstruction | NB-8 |
| Synthesis — keystone-join sanity (engagement × outcome) | NB-10 |

## Setup

In [ ]:
# --- Setup: parameterized scope ---
#
# Change WINDOW_START / WINDOW_END to re-run the whole notebook under a different
# cycle restriction. All downstream queries reference these via Python f-string
# interpolation into spark.sql() calls. ICP is a slicing dimension, NOT a filter
# (resolved scope decision).

from pyspark.sql import SparkSession
from pyspark.sql import functions as F

try:
    spark  # type: ignore[name-defined]
except NameError:
    spark = SparkSession.builder.getOrCreate()

# ===== Tweakable parameters =====
WINDOW_START = "2025-05-01"
WINDOW_END = "2027-01-01"  # exclusive (i.e. includes through 2026-12-31)
# ================================

# Prod schema paths (per memory: hit prod directly, not via ref())
CATALOG = "goodparty_data_catalog"
ANALYTICS = f"{CATALOG}.mart_analytics"
CIVICS = f"{CATALOG}.mart_civics"
DEV = f"{CATALOG}.private_tristan"
# IMPORTANT: most prod intermediate tables materialize to the `dbt` schema, not
# `dbt_intermediate`. `dbt_intermediate` does exist for some models but not all
# — if a table lookup errors, try `dbt` first.
DBT = f"{CATALOG}.dbt"
STAGING = f"{CATALOG}.dbt_staging"

WINDOW_CLAUSE = f"u.general_election_date BETWEEN '{WINDOW_START}' AND DATE_SUB('{WINDOW_END}', 1)"

print(f"Window: {WINDOW_START} (inclusive) to {WINDOW_END} (exclusive)")
print(f"WINDOW_CLAUSE: {WINDOW_CLAUSE}")

def show_count(df, label):
    n = df.count()
    print(f"{label}: {n:,}")
    return n

## NB-1 — Volume on `users_win_candidacy` (windowed)

**Maps to:** INVENTORY.md **Source 1.2 / Completeness summary**.

#### Context

`users_win_candidacy` (`dbt/project/models/marts/analytics/users_win_candidacy.sql`) is the keystone mart for this analysis: it already joins product user → candidacy → outcomes → viability → segmentation in a single denormalized row. Materialized as a view; grain is one row per `campaign_version_id` (campaign-version — a single product `campaign_id` may have multiple historical versions if the user reused the campaign across cycles).

For the EDA we filter to:
- `is_latest_version = true` — separates the canonical row from historical reuse versions. For "did the user win?" analyses, pre-filter unless explicitly studying campaign reuse.
- `NOT is_demo` — excludes test campaigns. Applied at the candidacy grain.
- `general_election_date BETWEEN '2025-05-01' AND '2026-12-31'` — the resolved cycle window.

#### Expected output (at default scope)

- Total non-demo latest-version rows: ~59,826
- **In-window non-demo latest-version rows: 26,072** (44% of total — this is the working population for the EDA)
- Distinct `user_id` in window: 25,982 (≈1:1 with rows — most users have a single in-window candidacy)
- Distinct `campaign_id` in window: 25,983

#### Why this matters

The cycle window slices the keystone mart in half (44% remain). All headline coverage stats throughout the inventory are reported against this in-window base — when you re-run with a different `WINDOW_START` / `WINDOW_END`, the headline numbers in subsequent cells will move proportionally. Total → in-window is the first verification step.

**Quality flags** (from INVENTORY Source 1.2):
- `is_latest_version` separates canonical rows from reuse versions (`users_win_candidacy.sql:43`).
- The join from `campaigns` → `candidacy` is version-aware: `campaign_id AND election_date = general_election_date` (with `election_date IS NULL` fallback). See `users_win_candidacy.sql:113-118`. Campaign-versions predating BR coverage surface as rows with NULL candidacy fields, not dropped rows.

In [ ]:
sdf = spark.sql(f"""
WITH u AS (
    SELECT *
    FROM {ANALYTICS}.users_win_candidacy
    WHERE is_latest_version AND NOT is_demo
)
SELECT
    COUNT(*) AS total_rows,
    COUNT(CASE WHEN general_election_date BETWEEN '{WINDOW_START}' AND DATE_SUB('{WINDOW_END}', 1) THEN 1 END) AS in_window_rows,
    COUNT(DISTINCT CASE WHEN general_election_date BETWEEN '{WINDOW_START}' AND DATE_SUB('{WINDOW_END}', 1) THEN user_id END) AS in_window_users,
    COUNT(DISTINCT CASE WHEN general_election_date BETWEEN '{WINDOW_START}' AND DATE_SUB('{WINDOW_END}', 1) THEN campaign_id END) AS in_window_campaigns
FROM u
""")

sdf.toPandas()

## NB-2 — Outcome and viability coverage on the in-window cohort

**Maps to:** INVENTORY.md **Source 1.2 / Completeness summary**.

### Context

Outcome variable per resolved scope (2026-05-27) is **`latest_stage_reached` + `latest_stage_result`** from `mart_civics.candidacy`, NOT `general_election_result`. The pair preserves the deepest stage a candidate reached and the result there, so candidates eliminated at primary still get a captured outcome.

`latest_stage_result` is joined into the keystone mart via:
```
users_win_candidacy.campaign_id ↔ candidacy.product_campaign_id
```

`viability_score` and `win_number` live directly on `users_win_candidacy` (passed through from `candidacy`).

### Expected output (at default scope)

| Field | Populated | % of in-window |
|---|---:|---:|
| `latest_stage_result` (any value) | 15,826 | **60.7%** |
| `general_election_result` (general stage only) | 11,329 | 43.5% |
| `viability_score` | 3,281 | **12.6%** |
| `win_number` | 13,897 | 53.3% |

### Why this matters

**`latest_stage_result` is populated 1.4× more often than `general_election_result`**. This is the empirical justification for the scope decision: candidates who lost at primary or earlier stages have a captured outcome that the general-only projection masks as NULL. Using `general_election_result` would drop ~28% of the labeled population on the floor.

**Viability coverage at 12.6%** is the operative number for Win-product analyses; see Source 4 / NB-7a for why this is small (Win users come in via the `gp_api` path which itself doesn't populate viability — they only get a score if the ER crosswalk lands them on a TS or TS+BR candidacy that DOES carry one).

**Domain values for `latest_stage_result`** (per `m_civics.yaml:543-602`): `Won`, `Lost`, `Runoff`, `Withdrew`, `Not on Ballot`. (`Cannot Determine` is excluded by the `accepted_values` test for stage results.)

For binary win/loss analyses, filter to `latest_stage_result IN ('Won', 'Lost')` — this is what the per-question synthesis in INVENTORY.md does.

In [ ]:
sdf = spark.sql(f"""
WITH u AS (
    SELECT u.*, c.latest_stage_reached, c.latest_stage_result
    FROM {ANALYTICS}.users_win_candidacy u
    LEFT JOIN {CIVICS}.candidacy c
      ON u.campaign_id = c.product_campaign_id
    WHERE u.is_latest_version AND NOT u.is_demo
      AND u.general_election_date BETWEEN '{WINDOW_START}' AND DATE_SUB('{WINDOW_END}', 1)
)
SELECT
    COUNT(*) AS rows_in_window,
    COUNT(latest_stage_result) AS latest_stage_result_populated,
    COUNT(general_election_result) AS general_election_result_populated,
    COUNT(viability_score) AS viability_populated,
    COUNT(win_number) AS win_number_populated,
    ROUND(100.0 * COUNT(latest_stage_result) / COUNT(*), 1) AS latest_pct,
    ROUND(100.0 * COUNT(general_election_result) / COUNT(*), 1) AS general_pct,
    ROUND(100.0 * COUNT(viability_score) / COUNT(*), 1) AS viability_pct,
    ROUND(100.0 * COUNT(win_number) / COUNT(*), 1) AS win_number_pct
FROM u
""")


sdf.toPandas().melt(var_name='variable', value_name='value')

## NB-3 — `candidacy_result` cross-stage fallback hazard

**Maps to:** INVENTORY.md **Source 1.5 (Outcome variables)** and **Biggest risks #9**.

### Context

`mart_civics.candidacy.candidacy_result` is a *rolled-up* outcome with cross-stage fallback (per `m_civics.yaml:516-545`):

```
candidacy_result = general_runoff > general > primary_runoff > primary
```

This means a candidate who **won their primary but lost their general** (or has no general yet captured) can show `candidacy_result = 'Won'` because the fallback dipped down to the primary stage. The same applies in reverse — `candidacy_result = 'Lost'` may reflect a primary loss for a candidate who never reached the general.

### What this query produces

A 2×N cross-tab of `candidacy_result` × `general_election_result` so you can see exactly how often the fallback diverges from the general-stage truth. Specific rows to look for:

- `candidacy_result = 'Won'` AND `general_election_result IS NULL` — primary winners whose general hasn't been ingested (or who lost in a stage between primary and general that didn't get logged)
- `candidacy_result = 'Won'` AND `general_election_result != 'Won'` — primary winners who lost the general (the contamination case)

### Why this matters

**For the resolved scope, prefer `latest_stage_reached + latest_stage_result` from `mart_civics.candidacy`** over `candidacy_result`. The latest-stage pair explicitly preserves both the deepest stage reached AND the result there, so you can decide per-analysis whether to filter to general-stage candidates only, or to count primary losses as outcomes.

`candidacy_result` is convenient for quick lookups but the cross-stage fallback contamination is documented but easy to miss — the data-quality risk for any binary win/loss analysis. This query quantifies the magnitude of the contamination so you can size the risk.

This cell runs against the full civics mart (unwindowed) because the fallback pattern is a global property of the mart, not a window-specific phenomenon.

In [ ]:
sdf = spark.sql(f"""
select
    coalesce(candidacy_result, 'NULL') as candidacy_result,
    coalesce(general_election_result, 'NULL') as general_election_result,
    count(*) as rows
from {CIVICS}.candidacy
group by 1, 2
order by 1, 2
""")

sdf.toPandas()

## NB-4 — Office-level distribution and outcome coverage by `election_level`

**Maps to:** INVENTORY.md **Source 1.4 (Segmentation dimensions)** and **Biggest risks #2**.

### Context

`election_level` is the primary office-level dimension for stratification: takes values `city`, `county`, `state`, `federal`, or NULL. The dimension matters for any "which actions correlate with winning" analysis because action effectiveness varies sharply by office tier (a single dashboard view ≠ a single dashboard view at federal level).

The query slices the in-window cohort by `election_level` and reports:
- Row count + share of in-window
- `viability_score` coverage %
- `latest_stage_result` coverage %
- `general_election_result` coverage %

### Expected output (at default scope)

| election_level | rows | % of in-window | viability % | latest_stage_result % | general_election_result % |
|---|---:|---:|---:|---:|---:|
| **NULL** | 11,891 | **45.6%** | 8.5% | 50.1% | 34.6% |
| city | 11,178 | 42.9% | 19.5% | 84.4% | 63.3% |
| state | 1,233 | 4.7% | 1.2% | 10.2% | 0.7% |
| federal | 897 | 3.4% | 0.1% | 1.8% | 0.1% |
| county | 873 | 3.3% | 7.8% | 32.6% | 15.1% |

### Why this matters

**⚠ 45.6% of in-window candidacies have NULL `election_level`.** That's the single largest bucket. Down from 61.5% in the unwindowed view but still dominant. The NULL bucket has meaningful outcome coverage (50% have `latest_stage_result`), so dropping it loses real signal.

**Three options before scoping any office-level slice:**
1. Restrict to non-NULL — cuts sample size from 26k to ~14k.
2. Backfill NULL from `office_type` (a different field on the candidacy, populated more often) or by joining `int__icp_offices`.
3. Treat NULL as a fifth category (and surface this choice in any downstream analysis).

**City is the dominant non-NULL bucket** (43% of in-window). The state/federal buckets each have <5% and effectively zero outcome coverage in window. Multi-level comparisons should expect city to dominate any pooled estimate.

### Other segmentation dimensions available

Also on `users_win_candidacy`, queryable via the same windowed-base pattern:
- `office_type` (string, HubSpot-sourced — common values documented at `m_civics.yaml:507-514`: School Board, City Council, Mayor, Judge, State House, etc.)
- `campaign_state`, `campaign_party`
- `is_pro` — see NB-6
- `icp_office_win`, `icp_office_serve`, `icp_win_supersize` — see NB-11
- `is_judicial`, `is_appointed`
- L2 district context: `l2_district_name`, `l2_district_type`, `voter_count`

Not on `users_win_candidacy` directly, requires a join to `mart_civics.candidacy`:
- `is_incumbent` (TS-sourced, ~70% populated; BR-only rows are typically NULL)
- `is_open_seat` (BR > TS > DDHQ — NULL on BR-only rows)
- `is_partisan` (boolean)

In [ ]:
sdf = spark.sql(f"""
WITH u AS (
    SELECT u.*, c.latest_stage_reached, c.latest_stage_result
    FROM {ANALYTICS}.users_win_candidacy u
    LEFT JOIN {CIVICS}.candidacy c
      ON u.campaign_id = c.product_campaign_id
    WHERE u.is_latest_version AND NOT u.is_demo
      AND u.general_election_date BETWEEN '{WINDOW_START}' AND DATE_SUB('{WINDOW_END}', 1)
)
SELECT
    COALESCE(election_level, 'NULL') AS election_level,
    COUNT(*) AS rows,
    ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 1) AS pct_of_in_window,
    ROUND(100.0 * COUNT(viability_score) / COUNT(*), 1) AS viability_pct,
    ROUND(100.0 * COUNT(latest_stage_result) / COUNT(*), 1) AS latest_pct,
    ROUND(100.0 * COUNT(general_election_result) / COUNT(*), 1) AS general_pct
FROM u
GROUP BY election_level
ORDER BY rows DESC
""")

sdf.toPandas()

## NB-5 — Cycle slice: 2025 vs 2026 outcome and viability coverage

**Maps to:** INVENTORY.md **Source 2.2 (Data quality and coverage)** and **Biggest risks #6**.

### Context

The civics mart `candidacy` table is built as a UNION ALL of two structurally different parts (see `candidacy.sql`):

1. **2025 archive** (`int__civics_candidacy_2025`): HubSpot-only. Inherits viability through the HubSpot company-id crosswalk to `stg_model_predictions__viability_scores`.
2. **2026+ FOJ** (`int__civics_candidacy_ballotready / techspeed / ddhq / gp_api`): A four-way full outer join across the four 2026+ providers, with ER crosswalk via `int__civics_er_canonical_ids`.

Coverage of fields differs across cycles. This slice quantifies the difference.

### Expected output (at default scope, joined to `mart_civics.candidacy` for `latest_stage_result`)

| cycle | rows | viability % | latest_stage_result % | general_election_result % | won (latest) | lost (latest) |
|---|---:|---:|---:|---:|---:|---:|
| 2025 | 16,280 | 16.9% | 91.8% | 66.4% | 7,541 | 5,293 |
| 2026 | 9,792 | 5.3% | 9.1% | 5.4% | 462 | 385 |

### Why this matters

**2025 is the operative cohort for outcomes**: 92% have a `latest_stage_result` and 66% have a `general_election_result`. Most 2025 races have concluded by the data refresh date.

**2026 is almost entirely unconcluded**: only 9.1% have any `latest_stage_result`. The few cases that do are early-2026 primaries that have already happened.

**Cross-cycle longitudinal risks**:
- Field availability differs (e.g., `is_open_seat` populated for BR > TS > DDHQ in 2026+; absent from 2025 archive).
- 2025 viability comes through the HubSpot crosswalk (TS-feature-keyed scoring), while 2026+ comes through the BR side of the 4-way FOJ — different join paths, potentially different score distributions even for similar candidates.
- The 2025 cohort dominates win/loss labels: **12,834 of the 13,681 in-window labeled outcomes (94%) are 2025 races**. Any cross-cycle analysis is effectively a 2025-anchored analysis with limited 2026 holdout.

**Source-system shape difference (qualitative):**
- 2025 archive rows: `source_systems = ['hubspot']` (plus `'ddhq'` if there's a DDHQ outcome match)
- 2026+ rows: `source_systems` is some subset of `{'ballotready', 'techspeed', 'ddhq', 'gp_api'}` — never includes `'hubspot'`

If your analysis needs to stratify by cycle, slice on `year(general_election_date)` AND `source_systems` together to see the structural mix. The NB-7a viability coverage breakdown by `source_systems` (see Source 4.3) is the canonical cross-section.

In [ ]:
sdf = spark.sql(f"""
with c as (
    select
        year(general_election_date) as election_year,
        sort_array(source_systems) as sources,
        viability_score,
        is_incumbent,
        is_open_seat,
        candidacy_result
    from {CIVICS}.candidacy
        where general_election_date BETWEEN '{WINDOW_START}' AND DATE_SUB('{WINDOW_END}', 1)   
)
select
    election_year,
    sources,
    count(*) as rows,
    count(viability_score) as viability_populated,
    count(is_incumbent) as incumbent_populated,
    count(is_open_seat) as open_seat_populated,
    count(candidacy_result) as candidacy_result_populated
from c
group by 1, 2
order by 1, 3 desc
""")

sdf.toPandas()

## NB-6 — Pro vs free outcome comparison (windowed, outcome = `latest_stage_result`)

**Maps to:** INVENTORY.md **Source 1.4** and **Biggest risks #3** + **Synthesis Q3**.

### Context

`is_pro` is the campaign-grain Pro flag on `users_win_candidacy` — derived upstream from `campaigns.is_pro` (the raw product DB column). It's the treatment variable for the "does Pro-tier show outcome lift?" analysis (Synthesis Q3).

This query slices the in-window cohort by `is_pro` and reports:
- Row count
- Viability coverage % (because Pro / free populations may differ in race competitiveness)
- Decision counts (rows with `latest_stage_result ∈ ('Won', 'Lost')` — the denominator for win-rate)
- Win-rate among rows with a decision

### Expected output (at default scope)

| is_pro | rows | viability % | with_decision | won | **win_rate_pct** |
|---|---:|---:|---:|---:|---:|
| false | 24,954 | 11.7% | 13,091 | 7,769 | **59.3%** |
| true | 1,086 | 32.0% | 586 | 233 | **39.8%** |
| NULL | 32 | 9.4% | 4 | 1 | 25.0% (noisy) |

### Why this matters — the counterintuitive finding

**⚠ Pro users show a LOWER raw win rate (39.8%) than free users (59.3%) in window.** This is the single most surprising headline from the scout pass — the naive "Pro = better outcomes" expectation flips. The earlier unwindowed numbers (46% Pro vs 64% free) showed a wider gap; in window the gap narrows to ~20 percentage points but the direction persists.

Three caveats before reading any causal signal into this:

1. **Selection bias.** Pro users self-select into higher-stakes races (state, federal, contested cities) where competition is fiercer. The 32% viability coverage for Pro users vs 12% for free users suggests Pro candidates are more likely to be in races where viability scoring is even computed (i.e., the model has enough opponent / incumbency context). Free users may be skewed toward low-competition local races (school board, town council) where the field is small or uncontested.

2. **Reverse causation on engagement.** Pro upgrade is downstream of high engagement — but high engagement may also be a proxy for "candidate took the campaign seriously" which is itself correlated with running in a tougher race. Without office stratification + pre/post-upgrade timing, the comparison is uninterpretable.

3. **Small Pro denominator.** The Pro bucket has 586 rows with a decision vs 13,091 for free users. Confidence intervals on 39.8% will be wide; binomial CI is roughly ±4pp at this sample size. Office-stratified subgroup analyses will have very wide CIs.

4. **Pro upgrade TIMING matters.** `is_pro` reflects the campaign-grain Pro flag at the time the mart materializes; it does NOT carry the upgrade timestamp. To answer "does upgrading PREDICT winning" (causally) vs "are Pro users different in some other way" (descriptively), you need `pro_upgrade_completed_at` from `int__amplitude_user_milestones` and the pre/post-upgrade engagement frame.

### Recommended next steps in the EDA (per Synthesis Q3)

- Stratify Pro vs free by `election_level` × `office_type` first — does the gap survive controls?
- Layer in `is_incumbent` (TS-sourced, from civics mart join). Incumbent free users likely dominate the easy-win bucket.
- Add `is_open_seat` similarly. Open-seat races are competitive by definition.
- Add `pro_upgrade_completed_at` to compare candidates who upgraded EARLY (predictor) vs LATE (downstream of being competitive).

In [ ]:
sdf = spark.sql(f"""
WITH u AS (
    SELECT u.*, c.latest_stage_reached, c.latest_stage_result
    FROM {ANALYTICS}.users_win_candidacy u
    LEFT JOIN {CIVICS}.candidacy c
      ON u.campaign_id = c.product_campaign_id
    WHERE u.is_latest_version AND NOT u.is_demo
      AND u.general_election_date BETWEEN '{WINDOW_START}' AND DATE_SUB('{WINDOW_END}', 1)
)
SELECT
    COALESCE(CAST(is_pro AS STRING), 'NULL') AS is_pro,
    COUNT(*) AS rows,
    COUNT(viability_score) AS viability_populated,
    ROUND(100.0 * COUNT(viability_score) / COUNT(*), 1) AS viability_pct,
    COUNT(CASE WHEN latest_stage_result IN ('Won', 'Lost') THEN 1 END) AS with_decision,
    COUNT(CASE WHEN latest_stage_result = 'Won' THEN 1 END) AS won,
    ROUND(100.0 * COUNT(CASE WHEN latest_stage_result = 'Won' THEN 1 END)
          / NULLIF(COUNT(CASE WHEN latest_stage_result IN ('Won', 'Lost') THEN 1 END), 0), 1) AS win_rate_pct
FROM u
GROUP BY 1
ORDER BY rows DESC
""")

sdf.toPandas()

## NB-11 — ICP slice (dimension, not filter)

**Maps to:** INVENTORY.md **Source 1.4 (Segmentation dimensions)**, **Synthesis Q5 (Are there meaningful user segments with distinct outcome patterns?)**, and **Biggest risks #3**.

### Context (resolved scope)

`icp_office_win` flags whether a candidacy is for an office Win supports (the "ideal customer profile" for the Win product). It was originally proposed as a population filter (restrict to ICP=true) but the resolved scope (2026-05-27) says: **use as a slicing dimension, NOT a filter**.

Rationale for slicing rather than filtering:
- ICP=true candidacies are competitive races by definition (Win supports offices where the campaign matters). Filtering removes the comparison baseline.
- Reporting unfiltered first characterizes the broader Win population; ICP slices show whether the patterns differ for the ICP-aligned subset.
- Same logic applies for `icp_office_serve` and `icp_win_supersize` (other ICP flags), though we don't slice those here.

This cell reports per ICP bucket: row count, viability coverage %, latest_stage_result coverage %, decision count, and win rate.

### Expected output (at default scope)

| icp_office_win | rows | viability % | latest_pct | with_decision | **win_rate_pct** |
|---|---:|---:|---:|---:|---:|
| true | 12,427 | 11.0% | 64.4% | 6,857 | **53.8%** |
| false | 11,030 | 12.0% | 50.8% | 4,783 | **63.5%** |
| NULL | 2,615 | 22.6% | 85.0% | 2,041 | 62.6% |

### Why this matters — the ICP-true cohort has a LOWER raw win rate

ICP=true candidates win less often (53.8%) than ICP=false (63.5%). Direction matches the Pro-vs-free finding from NB-6:

- **ICP=true cohort**: 12,427 rows, 53.8% win rate among decisions
- **ICP=false cohort**: 11,030 rows, 63.5% win rate among decisions
- **NULL cohort**: 2,615 rows, 62.6% win rate (most have high `latest_pct` of 85% — likely candidacies without a BR position match, so no ICP context)

Likely interpretations (all to be tested in the EDA, not concluded here):

1. **ICP=true offices are more competitive by design.** Win supports offices where there's a campaign worth running — i.e., offices with active opposition, contested races, or meaningful electorate. Non-ICP offices may be dominated by uncontested low-stakes races (uncontested local board, school commissioner with no opponent).

2. **Selection effect.** Candidates who choose Win-supported offices may be self-selected for harder races. Non-ICP candidates may be running in races where winning is more about showing up than campaigning.

3. **NULL ICP carries different population.** The 2,615 NULL rows have suspiciously high `latest_pct` of 85% — they may be a specific cohort (e.g., candidacies that never got a BR position match because they're in micro-races, where outcomes ARE captured because they're trivially uncontested).

### How to use this slice in the EDA

- For Synthesis Q5 ("meaningful user segments with distinct outcome patterns"), ICP × `election_level` is a useful 2D slice. Expect ICP=true to skew toward city / state races; non-ICP toward less-classified offices.
- For Synthesis Q3 (Pro lift), ICP should be a control alongside office type. Pro users may overlap with ICP=true candidates (same self-selection into competitive races).
- The NULL bucket should be reported separately; combining with false ICP risks hiding a structural data gap.

### Related dimensions also available

- `icp_office_serve` — Serve-product ICP. Similar logic, narrower (Serve has a smaller candidate base).
- `icp_win_supersize` — flags large-electorate offices that are outside the standard Win process. Use to exclude super-large races if needed.
- `is_judicial`, `is_appointed` — flag non-traditional offices. Often correlated with NULL ICP.

In [ ]:
sdf = spark.sql(f"""
WITH u AS (
    SELECT u.*, c.latest_stage_reached, c.latest_stage_result
    FROM {ANALYTICS}.users_win_candidacy u
    LEFT JOIN {CIVICS}.candidacy c
      ON u.campaign_id = c.product_campaign_id
    WHERE u.is_latest_version AND NOT u.is_demo
      AND u.general_election_date BETWEEN '{WINDOW_START}' AND DATE_SUB('{WINDOW_END}', 1)
)
SELECT
    COALESCE(CAST(icp_office_win AS STRING), 'NULL') AS icp_office_win,
    COUNT(*) AS rows,
    ROUND(100.0 * COUNT(viability_score) / COUNT(*), 1) AS viability_pct,
    ROUND(100.0 * COUNT(latest_stage_result) / COUNT(*), 1) AS latest_pct,
    COUNT(CASE WHEN latest_stage_result IN ('Won', 'Lost') THEN 1 END) AS with_decision,
    ROUND(100.0 * COUNT(CASE WHEN latest_stage_result = 'Won' THEN 1 END)
          / NULLIF(COUNT(CASE WHEN latest_stage_result IN ('Won', 'Lost') THEN 1 END), 0), 1) AS win_rate_pct
FROM u
GROUP BY 1
ORDER BY rows DESC
""")

sdf.toPandas()

## NB-8 — Recommended-action funnel: does the product DB carry structured task data?

**Maps to:** INVENTORY.md **Source 3.8 (Recommended-action funnel)** and **Source 5.4 (Product DB segmentation)**.

### Context

The user's original question included "what fraction of recommended actions do users complete, and where's the drop-off?" The Amplitude-side answer (Source 3.8) is that the question IS answerable — instrumentation is rich across 15+ Win-product feature areas; new aggregates over raw `stg_airbyte_source__amplitude_api_events` are needed.

This cell complements that by inspecting the **product DB side** to see whether there is also a structured task / step / recommended-action schema we could use instead (or alongside).

Tables inspected:
- `stg_airbyte_source__gp_api_db_path_to_victory` — name suggests a strategy / action-plan table
- `stg_airbyte_source__gp_api_db_outreach` — voter outreach campaign records
- `stg_airbyte_source__gp_api_db_campaign` — campaign rows with a `details` JSON field

For each table, the cell prints the schema and 3 sample rows so you can eyeball:
- Is there a per-user task / step / completion column?
- Is there a foreign key to a "recommended action" template table?
- Is there a status / state field that would indicate completion?

### Why this matters

If `path_to_victory` carries structured task data, **the recommended-action funnel becomes answerable from the product DB alone** (with Amplitude only for completion timestamps). That's a simpler integration than building new Amplitude aggregates.

If it doesn't, the Amplitude-aggregates path (Source 3.8) is the way — confirmed answerable, just needs the work.

### Open questions for the EDA, contingent on what this cell shows

1. Does `path_to_victory` have a task/step schema or is it a single denormalized strategy blob?
2. Does the product DB write back from the `Candidacy - Did You Win Modal` Amplitude event? (Related: NB-9c flags this as a question — if the answer is on the candidacy row, it expands the outcome label set materially.)
3. Are there other product-DB tables (campaign_planner_step, recommended_actions, candidate_tasks, ...) we haven't enumerated?

This is a schema-introspection cell, not a coverage analysis. Output is sample rows for visual judgment, not a single headline number.

In [ ]:
# --- NB-7a: viability coverage by source bucket (mart_civics.candidacy) ---
sdf = spark.sql(f"""
SELECT
    CASE
        WHEN array_contains(source_systems, 'hubspot') THEN '2025 archive (hubspot)'
        WHEN array_contains(source_systems, 'ballotready') THEN '2026+ BR-matched'
        WHEN array_contains(source_systems, 'techspeed') THEN '2026+ TS-only'
        WHEN array_contains(source_systems, 'gp_api') THEN '2026+ gp_api-only'
        WHEN array_contains(source_systems, 'ddhq') THEN '2026+ ddhq-only'
        ELSE 'other'
    END AS source_bucket,
    COUNT(*) AS rows,
    COUNT(viability_score) AS viability_populated,
    ROUND(100.0 * COUNT(viability_score) / COUNT(*), 2) AS pct,
    COUNT(CASE WHEN general_election_date BETWEEN '2025-05-01' AND '2026-12-31' THEN 1 END) AS in_window,
    COUNT(CASE WHEN general_election_date BETWEEN '2025-05-01' AND '2026-12-31' THEN viability_score END) AS in_window_rated
FROM {CIVICS}.candidacy
WHERE general_election_date IS NOT NULL
GROUP BY 1
ORDER BY rows DESC
""")

sdf.toPandas()

In [ ]:
# --- NB-7a: TS-side scoring intermediate (the source of truth for the score) ---
sdf = spark.sql(f"""
SELECT
    COUNT(*) AS rows,
    COUNT(viability_rating_2_0) AS rated,
    COUNT(DISTINCT techspeed_candidate_code) AS distinct_candidates,
    AVG(viability_rating_2_0) AS avg_rating,
    MIN(viability_rating_2_0) AS min_rating,
    MAX(viability_rating_2_0) AS max_rating
FROM {CATALOG}.dbt.int__techspeed_viability_scoring
""")

sdf.toPandas()

In [ ]:
# --- NB-7a: score-band distribution ---
sdf = spark.sql(f"""
SELECT score_viability_automated AS band, COUNT(*) AS rows
FROM {CATALOG}.dbt.int__techspeed_viability_scoring
GROUP BY 1
ORDER BY 2 DESC
""")

sdf.toPandas()

In [ ]:
# --- NB-7a: model_predictions.viability_scores (HubSpot-keyed scoring run) ---
sdf = spark.sql(f"""
SELECT
    COUNT(*) AS rows,
    COUNT(viability_rating_2_0) AS rated,
    COUNT(DISTINCT id) AS distinct_hubspot_company_ids,
    AVG(viability_rating_2_0) AS avg_rating
FROM {CATALOG}.dbt.stg_model_predictions__viability_scores
""")

sdf.toPandas()

## NB-9 — Amplitude event-type inventory: full universe vs modeled subset

**Maps to:** INVENTORY.md **Source 3.1a (Universe vs modeled)**, **Source 3.8 (Recommended-action funnel)**, **Biggest risks #1**.

### Context

The dbt intermediate Amplitude models cover only a small slice of what is actually being tracked. Counting distinct `event_type` values within the analysis window (`event_time` between `WINDOW_START` and `WINDOW_END`):

- **Total distinct event types in window:** 314 (at default scope)
- **Aggregated in at least one intermediate model:** 12 (3.8% of event types)
- **Events on modeled types:** 291,478 of 12,363,089 total events (2.4%)

The modeled subset:
- `int__amplitude_win_activity` aggregates 2 event types: `Voter Outreach - Campaign Completed` and `Dashboard - Candidate Dashboard Viewed`
- `int__amplitude_user_milestones` aggregates ~12 lifecycle milestones (registration, dashboard view, onboarding complete, campaign sent, pro upgrade, plus 6 Serve onboarding milestones)
- `int__amplitude_serve_activity` filters the generic `Viewed` event to `event_properties:path = '/dashboard/polls'` + `user_properties:Serve Activated = true` (partial use of `Viewed`)

Most bulk-event volume is **auto-instrumented Amplitude system events** with no `user_id` attached (`[Amplitude] Page Viewed`, `Scroll Depth`, `session_start`). If we restrict to candidate-attributed product events (excluding `amplitude_autotrack`, `experiment_assignment`, `gtm_bootstrap`, `session_or_browser`, `viewed_generic`, `marketing_or_misc`, `cta_clicks`):

- **Candidate-attributed events in window:** ~1.4M
- **Modeled candidate-attributed events:** 291k (~21%)

### What this cell produces

**NB-9a (family rollup):** Groups all 314 event types into 29 product/system families via SQL CASE-WHEN patterns. Reports per family: count of distinct event types, total events, distinct users (max within family), and count of modeled event types in the family.

**NB-9b (per-event breakdown):** A flat per-event_type listing with event count, distinct users, first_seen, last_seen, family, and modeled flag. Output is paginated in the notebook view; the full CSV is written locally by `_pull_amplitude_universe.py` (gitignored).

### Expected output: top families by event volume (at default scope)

| event_family | n_types | events | max_users_one_event | n_modeled | Notes |
|---|---:|---:|---:|---:|---|
| `viewed_generic` | 1 | 3.5M | 23,699 | 0* | `Viewed` event. Partially used by `int__amplitude_serve_activity` via property filter. |
| `amplitude_autotrack` | 12 | 3.0M | 0 | 0 | `[Amplitude] *` autoevents. Anonymous. Skip. |
| `session_or_browser` | 8 | 2.4M | 455 | 0 | `Scroll Depth`, `session_*`. Mostly anonymous. Skip. |
| `experiment_assignment` | 4 | 1.9M | 12,152 | 0 | `[Experiment] *`. Usable as a stratification covariate. |
| **`win_onboarding`** | 26 | 292k | 17,332 | 2 | Onboarding funnel. We model registration + `onboarding_complete`. 24 mid-funnel events unmodeled (largest: `Onboarding - Candidate Office Searched`, 17,332 users). |
| **`win_dashboard`** | 8 | 254k | 11,589 | 1 | We model `Dashboard - Candidate Dashboard Viewed`. New: `Dashboard - Campaign Plan Viewed` (started 2026-04, AI weekly plan). |
| **`win_outreach_scheduling`** | 15 | 252k | 2,048 | 0 | Schedule Text Campaign funnel — 15 steps. **Entirely unmodeled.** |
| `navigation` | 22 | 252k | 5,811 | 0 | Site navigation clicks. Useful for path analysis if needed. |
| **`win_content_builder`** | 20 | 64k | 2,331 | 0 | Content / AI authoring funnel. |
| **`win_voter_data`** | 24 | 54k | 1,515 | 0 | Voter file viewing, downloading, customizing. |
| **`win_outreach_planning`** | 6 | 46k | 4,024 | 0 | `Outreach - View Accessed`, `Outreach - Click Create`, door-knock + social-media actions. |
| **`win_pro_upgrade`** | 25 | 45k | 3,686 | 1 | We model `pro_upgrade_complete`. 24 mid-funnel events unmodeled (including `Pro Upgrade - Modal: Modal Shown`, 3,686 users — every user who saw the upsell). |
| **`win_candidate_profile`** | 20 | 27k | 2,161 | 0 | Profile setup — running against / top issues / campaign details / fun facts. |
| **`serve`** | 33 | 26k | 1,829 | 7 | Serve product. 7 onboarding events modeled. Unmodeled: `Serve Onboarding - Meet Your Constituents Viewed` (1,713 users), Sworn In events, polls authoring. |
| **`win_candidate_website`** | 8 | 11k | 1,516 | 0 | Public candidate-website builder. |
| **`win_voter_outreach`** | 8 | 11k | 1,240 | 1 | We model `Voter Outreach - Campaign Completed`. Unmodeled: 10DLC compliance funnel for outreach. |
| **`win_ai_assistant`** | 10 | 10k | 1,482 | 0 | Campaign assistant. |
| **`win_candidacy_self_report`** | 3 | 6k | 2,281 | 0 | **`Candidacy - Did You Win Modal` — see NB-9c, expands outcome label set.** |
| **`win_briefings`** | 6 | 242 | 8 | 0 | Brand-new feature (started 2026-04-10). Tiny user base. |
| (remaining 13 smaller families) | ... | ... | ... | ... | See NB-9b output for the full list. |

\* `viewed_generic` n_modeled = 0 strictly, but `int__amplitude_serve_activity` uses a property-filtered slice of `Viewed`.

### Top unmodeled events per Win family, ranked by `distinct_users`

(These are events we'd most want for a "what actions correlate with outcomes" analysis. The CSV from `_pull_amplitude_universe.py` has the full list.)

- **`win_onboarding`**: `Onboarding - Candidate Office Searched` (17,332u, 129k events) — the largest single candidate-touched event in the window; `Onboarding - Office Step: Office Selected` (12,758u); `Onboarding - Office Step: Click Next` (12,204u); `Onboarding - Party Step: Click Submit` (11,902u); `Onboarding - Candidate Office Completed` (11,794u)
- **`win_outreach_scheduling`**: `Schedule Text Campaign: Exit` (2,048u); `Schedule Text Campaign: Next` (1,851u); `Schedule Text Campaign - Audience: Check Audience` (1,332u)
- **`win_pro_upgrade`**: `Pro Upgrade - Modal: Modal Shown` (3,686u — denominator for upsell impressions); `Pro Upgrade - Modal: Exit` (2,768u); `Pro Upgrade: Confirm office` (2,329u); `Pro Upgrade - Splash Page: Click upgrade` (2,161u)
- **`win_content_builder`**: `Content Builder: Generation Started` (2,331u); `Content Builder: Generation Completed` (2,317u); `Content Builder: Click Generate` (2,178u)
- **`win_outreach_planning`**: `Outreach - View Accessed` (4,024u); `Outreach - Click Create` (2,605u)
- **`win_candidate_profile`**: `Profile - Running Against: Click Save` (2,161u); `Profile - Top Issues: Click Finish Entering Issues` (1,595u)
- **`win_ai_assistant`**: `question_complete` (1,482u); `AI Assistant: Ask a question` (985u)
- **`win_voter_data`**: `Voter Data: Click Detail View` (1,515u); `Download Voter File attempt` (1,212u)
- **`win_candidate_website`**: `Candidate Website - Started` (1,516u); `Candidate Website - Published` (1,267u)
- **`win_candidacy_self_report`**: `Candidacy - Did You Win Modal Viewed` (2,281u, since 2025-10-31); `Candidacy - Did You Win Modal Completed` (2,159u) — see NB-9c

### Instrumentation start dates

Most product-event families have first-seen dates clustered around **2025-05-28** (likely a major instrumentation push around that date). Briefings (2026-04-10), Candidacy self-report (2025-10-31), and Dashboard Campaign Plan (2026-04-09) are newer. Anything before 2025-05-28 was tracked sparsely.

### Implications for the analysis

1. **The "what fraction of recommended actions do users complete" question is answerable.** Required work is new intermediate aggregates over the raw event stream, not new instrumentation. See Source 3.8.
2. **The current Amplitude intermediate models vastly undersample available signal.** Adding event-family aggregates is a tractable in-scope build using the FAMILY_CASE_SQL pattern below.
3. **`Candidacy - Did You Win` is a previously-unsurfaced outcome stream.** See NB-9c.

In [ ]:
# Schema introspection on path_to_victory + adjacent product tables.
# Adjust schema name if your dbt staging schema is different.
for tbl in (
    "stg_airbyte_source__gp_api_db_path_to_victory",
    "stg_airbyte_source__gp_api_db_outreach",
    "stg_airbyte_source__gp_api_db_campaign",
):
    print(f"=== {tbl} ===")
    try:
        df = spark.table(f"{CATALOG}.dbt_staging.{tbl}")
        df.printSchema()
        df.limit(3).show(truncate=80)
    except Exception as e:
        print(f"(could not load: {e})")
    print()

## NB-9 — Amplitude event-type inventory: full universe vs modeled subset

Maps to INVENTORY.md "Source 3.1a Universe vs modeled".

The intermediate Amplitude models (`int__amplitude_win_activity`, `int__amplitude_user_milestones`, `int__amplitude_serve_activity`) collectively aggregate **12 distinct event_types**. The raw staging table `stg_airbyte_source__amplitude_api_events` carries **314 distinct event_types** in the analysis window (2025-05-01 through 2026-12-31). The two cells below build the family rollup and produce a CSV of the full per-event breakdown.

The per-event CSV is written to the local notebook folder and gitignored (covered by `analytics/*/notebooks/**`).

In [ ]:
# --- NB-9a: family rollup of all Amplitude event_types in window ---
#
# Groups raw event_types into product feature areas via CASE-WHEN patterns,
# then counts events / distinct users / modeled-event-type fraction per family.
# Family definitions are duplicated in _pull_amplitude_universe.py.

FAMILY_CASE_SQL = """
CASE
    WHEN event_type LIKE '[Amplitude]%' THEN 'amplitude_autotrack'
    WHEN event_type LIKE '[Experiment]%' OR event_type = 'Experiment Viewed' THEN 'experiment_assignment'
    WHEN event_type LIKE 'gtm.%' THEN 'gtm_bootstrap'
    WHEN event_type IN ('Scroll Depth', 'session_start', 'session_end', 'page_view', 'Page Viewed', 'Page', 'usersnap_submission') OR event_type LIKE 'Segment Consent%' THEN 'session_or_browser'
    WHEN event_type LIKE 'Viewed%/elections%' OR event_type IN ('likelihood-to-cancel', 'Daily Ad Metrics') OR event_type LIKE '[AI Visibility]%' OR event_type LIKE 'AI Visibility%' THEN 'marketing_or_misc'
    WHEN event_type LIKE 'Onboarding -%' OR event_type LIKE 'Onboarding:%' OR event_type IN ('onboarding_complete', 'Invalid Party', 'Sign Up Clicked') THEN 'win_onboarding'
    WHEN event_type LIKE 'Dashboard -%' THEN 'win_dashboard'
    WHEN event_type LIKE 'Voter Outreach -%' THEN 'win_voter_outreach'
    WHEN event_type LIKE 'Outreach -%' THEN 'win_outreach_planning'
    WHEN event_type LIKE 'Schedule Text Campaign%' OR event_type LIKE 'schedule_campaign%' THEN 'win_outreach_scheduling'
    WHEN event_type LIKE 'Content Builder%' OR event_type LIKE 'ai_content_%' OR event_type LIKE 'campaign_assistant%' THEN 'win_content_builder'
    WHEN event_type LIKE 'Voter Data -%' OR event_type LIKE 'Voter Data:%' OR event_type LIKE 'Download Voter%' OR event_type LIKE 'Custom Voter%' THEN 'win_voter_data'
    WHEN event_type LIKE 'Profile -%' THEN 'win_candidate_profile'
    WHEN event_type LIKE 'Pro Upgrade -%' OR event_type LIKE 'Pro Upgrade:%' OR event_type = 'pro_upgrade_complete' OR event_type LIKE 'Pro user can not%' THEN 'win_pro_upgrade'
    WHEN event_type LIKE 'P2P Upgrade -%' THEN 'win_p2p_upgrade'
    WHEN event_type LIKE 'Candidate Website%' THEN 'win_candidate_website'
    WHEN event_type LIKE 'Candidacy -%' THEN 'win_candidacy_self_report'
    WHEN event_type LIKE 'Campaign Verify%' OR event_type LIKE 'Campaign Plan%' OR event_type LIKE '10 DLC Compliance%' THEN 'win_compliance_or_planning'
    WHEN event_type LIKE 'AI Assistant%' OR event_type = 'question_complete' THEN 'win_ai_assistant'
    WHEN event_type LIKE 'Briefings -%' THEN 'win_briefings'
    WHEN event_type LIKE 'Contacts -%' THEN 'win_contacts'
    WHEN event_type LIKE 'Resources -%' THEN 'win_resources'
    WHEN event_type LIKE 'Serve Onboarding%' OR event_type LIKE 'Poll - %' OR event_type LIKE 'Polls -%' OR event_type LIKE 'Polls:%' THEN 'serve'
    WHEN event_type LIKE 'Sign In:%' OR event_type LIKE 'Sign Up:%' OR event_type LIKE 'Set Password:%' OR event_type LIKE 'Account -%' OR event_type LIKE 'Settings -%' THEN 'auth_or_settings'
    WHEN event_type LIKE 'Navigation -%' OR event_type LIKE 'Navigation Top -%' THEN 'navigation'
    WHEN event_type LIKE 'Payment -%' THEN 'payment'
    WHEN event_type LIKE 'Click to Call%' OR event_type LIKE 'candidate CTA%' THEN 'cta_clicks'
    WHEN event_type = 'Viewed' THEN 'viewed_generic'
    ELSE 'other'
END
"""

MODELED_EVENT_TYPES_SQL = """(
    'Voter Outreach - Campaign Completed',
    'Dashboard - Candidate Dashboard Viewed',
    'Onboarding - Registration Completed',
    'onboarding_complete',
    'pro_upgrade_complete',
    'Serve Onboarding - SMS Poll Sent',
    'Serve Onboarding - Getting Started Viewed',
    'Serve Onboarding - Constituency Profile Viewed',
    'Serve Onboarding - Poll Value Props Viewed',
    'Serve Onboarding - Poll Strategy Viewed',
    'Serve Onboarding - Add Image Viewed',
    'Serve Onboarding - Poll Preview Viewed'
)"""

sdf = spark.sql(f"""
with classified as (
    select user_id, event_type, event_time,
           {FAMILY_CASE_SQL} as event_family,
           case when event_type in {MODELED_EVENT_TYPES_SQL} then true else false end as is_modeled
    from {CATALOG}.dbt_staging.stg_airbyte_source__amplitude_api_events
    where event_time >= '2025-05-01' and event_time < '2027-01-01'
)
select event_family,
       count(distinct event_type) as n_event_types,
       count(*) as events,
       count(distinct user_id) as distinct_users,
       count(distinct case when is_modeled then event_type end) as n_event_types_modeled
from classified
group by event_family
order by events desc
""")

sdf.toPandas()

In [ ]:
# --- NB-9b: per-event-type breakdown, full universe (saved to CSV) ---
#
# Writes the untruncated per-event breakdown to amplitude_event_universe.csv
# (gitignored) for use in building top-N-per-family call-outs in INVENTORY.md.

per_event_df = spark.sql(f"""
select event_type,
       {FAMILY_CASE_SQL} as event_family,
       count(*) as events,
       count(distinct user_id) as distinct_users,
       min(event_time) as first_seen,
       max(event_time) as last_seen,
       case when event_type in {MODELED_EVENT_TYPES_SQL} then true else false end as is_modeled
from {CATALOG}.dbt_staging.stg_airbyte_source__amplitude_api_events
where event_time >= '2025-05-01' and event_time < '2027-01-01'
group by event_type
order by events desc
""")

# In a Databricks notebook context, display() renders the full table.
# When running via Databricks Connect locally, .toPandas() + .to_csv() lets you
# inspect untruncated names locally; mirrors what _pull_amplitude_universe.py does.
# per_event_df.show(50, truncate=False)
per_event_df.toPandas()

## NB-9c — Did You Win Modal payload: schema + cohort + reconciliation with civics mart

**Maps to:** INVENTORY.md **Source 3.5 (Outcome variables)** + **Completeness summary (Outcomes — Amplitude supplement row)**.

#### Why this is the most important finding for outcomes work

Source 3.5 of INVENTORY.md was originally written as "n/a — Amplitude carries engagement, not electoral outcomes." That was wrong. The `win_candidacy_self_report` Amplitude family carries **self-reported electoral outcomes captured in product**, joinable by `user_id` to `users_win_candidacy`. This is a previously-unsurfaced outcome stream and a material expansion of the labeled-outcome population.

The three relevant event types (from `stg_airbyte_source__amplitude_api_events`):

| event_type | distinct_users | events | First seen |
|---|---:|---:|---|
| `Candidacy - Did You Win Modal Viewed` | 2,281 | 3,350 | 2025-10-31 |
| `Candidacy - Did You Win Modal Completed` | 2,159 | 2,434 | 2025-10-31 |
| `Candidacy - Campaign Completed` | 155 | 167 | 2025-09-18 |

#### Verified payload schema

The `event_properties` JSON on `Candidacy - Did You Win Modal Completed`:

```json
{"impersonation": <bool | null>, "status": "won" | "lost"}
```

- **`status`** is the candidate's self-reported outcome — string, either `"won"` or `"lost"`. No other values observed.
- **`impersonation`** indicates whether a staff member triggered the event while impersonating a user. The field was **added 2026-03**; events before that carry `impersonation = null`. For analytical use, treat `null` as not-impersonated (the legacy cohort is the bulk — 2,007 of 2,160 users).

The cell extracts these via `event_properties:status::string` and `event_properties:impersonation::boolean` (Databricks SQL VARIANT syntax).

#### What this cell produces

**NB-9c.1 (payload schema + impersonation field history):** Cross-tab of impersonation flag × event volume, showing the field-introduction date.

**NB-9c.2 (user-grain summary):** Latest non-impersonated response per `user_id`. Reports total users, the non-impersonated subset, and self-reported won/lost counts.

**NB-9c.3 (reconciliation with civics mart):** Cross-tab of Amplitude self-reported `status` × civics-mart `general_election_result` × `latest_stage_result` × `latest_stage_reached`. Shows where the two sources agree, disagree, or where one source has a label the other doesn't.

#### Expected output 1: impersonation field history

| impersonation_flag | first_seen_month | last_seen_month | events | distinct_users |
|---|---|---|---:|---:|
| `False` | 2026-03 | 2026-05 | 211 | 130 |
| `NULL (legacy/pre-field)` | 2025-10 | 2026-03 | 2,187 | 2,007 |
| `True` | 2026-03 | 2026-05 | 36 | 23 |

The field was added 2026-03. Filter `COALESCE(impersonation, false) = false` to drop impersonated events while keeping the legacy NULL cohort.

#### Expected output 2: user-grain summary (latest non-impersonated response per user)

| | users |
|---|---:|
| Total users with completed modal | 2,159 |
| With non-impersonated latest response | 2,136 |
| ↳ self-reported "won" | 1,707 (80%) |
| ↳ self-reported "lost" | 429 (20%) |

#### ⚠ Response selection bias is severe

The **80% self-reported win rate is far above the 64% raw win rate** among all candidacies with a `general_election_result` (Source 1.4). Two plausible drivers:
1. Winners are more motivated to come back to the product and respond.
2. The modal may be shown more aggressively to winners.

**The cohort is NOT a random sample of Win candidates.** Use the labels themselves to study correlates of winning *within the responding cohort*, but don't use the rate as a population-win-rate estimate.

#### Expected output 3: reconciliation with civics mart (top rows)

| amplitude_status | mart_general | latest_stage_result | latest_stage_reached | users |
|---|---|---|---|---:|
| won | Won | Won | general | 1,082 |
| won | NULL | Won | general | 469 |
| won | NULL | NULL | NULL | 127 |
| lost | NULL | Lost | general | 163 |
| lost | NULL | NULL | NULL | 135 |
| lost | Lost | Lost | general | 119 |
| won | Runoff | Runoff | general | 16 |
| won | NULL | Runoff | general | 10 |
| won | Lost | Lost | general | 9 ⚠ DISAGREE |
| lost | NULL | Won | general | 6 ⚠ DISAGREE |
| lost | NULL | Not on Ballot | general | 24 |
| won | NULL | Not on Ballot | general | 7 |
| (remaining smaller rows; see query output) | | | | |

#### Reconciliation summary

| Civics-mart state | Users | Notes |
|---|---:|---|
| `general_election_result` populated AND matches Amplitude | ~1,201 | Reconciled both ways. |
| `general_election_result` populated AND disagrees with Amplitude | ~17 | Disagreement (<1%) — manual investigation needed; likely user error, multi-cycle confusion, or modal interpretation issue. |
| `general_election_result` NULL but `latest_stage_result` populated | ~708 | **Mart partial — Amplitude adds full outcome.** Most are primary-stage losses or wins reported with no general stage ingested. |
| `general_election_result` NULL AND `latest_stage_result` NULL (no civics-mart row) | ~262 | **Amplitude-only outcome.** Users without a civics-mart candidacy join (could be edge-case `campaign_id` mismatch or candidacy not ingested). |

#### Net new outcome labels addressable from Amplitude

**~970 users** have an outcome label from Amplitude that is NOT available from `general_election_result`. This nearly doubles the labeled-outcome population (from 14,279 civics-mart general results to ~15,200 with Amplitude added).

#### Caveats for combining sources

- The modal asks the candidate about the general outcome but no formal authority order has been defined. The 17-user disagreement set should be inspected before adopting an authoritative-mart-then-Amplitude-fallback rule.
- Multi-cycle candidates may answer the modal in a different cycle than the one we're attributing to them — verify with `event_time` vs candidacy `general_election_date` if needed.
- The modal isn't required — only candidates who actively responded show up. Engagement-zero candidates by definition don't.

#### Implications for the EDA

- Use the labels (won/lost) themselves to study correlates of winning within the responding cohort.
- Do NOT use the cohort's 80% win rate as a population estimate.
- Layer Amplitude outcomes BENEATH civics-mart outcomes (mart authority on ties; Amplitude fills NULL cases).
- The 17-user disagreement set is small enough to manually inspect — that should resolve the authority-order question before scaling the join.

## NB-10 — Keystone join sanity: engagement × outcome (windowed)

**Maps to:** INVENTORY.md **Synthesis Q1** ("which product actions correlate with winning") and the per-join completeness row in the **Completeness summary**.

### Context

Synthesis Q1 is the headline analytical question: "which product actions correlate with winning, controlling for office type and level?" The keystone join is:

```
users_win_candidacy (cohort + segmentation)
  ↔ candidacy.product_campaign_id (outcomes — latest_stage_result, etc.)
  ↔ int__amplitude_win_activity_weekly (engagement, per-week)
```

All on `user_id` / `campaign_id` / `product_campaign_id`. The completeness summary reports the per-join success rates:
- Win candidate → engagement: ~96% (most Win users match an Amplitude user)
- Win candidate → outcomes: ~98% (most Win users land on a candidacy)

This cell pulls a 50-row sample restricted to Won/Lost candidacies in window so you can:
1. Visually confirm the join works end-to-end (a single row carries cohort + outcome + engagement).
2. See the shape of the engagement columns (`total_events`, `activity_days`, `campaigns_sent`, `dashboard_views`).
3. Spot any obvious join pathologies (rows with NULL engagement when they should have data, or duplicated outcomes).

### Why this matters

A working sample-row output is the precondition for the EDA. If the join produces 0 rows, or produces rows with all-NULL engagement, the EDA can't proceed — debug before scoping further.

The 50-row sample is intentionally small. For volume sanity at scale, look at NB-1 and NB-2's counts; for engagement-distribution shape, build a separate distribution cell once the EDA proper starts.

### Notable about this sample

- Restricted to `latest_stage_result IN ('Won', 'Lost')` — the binary outcome subset.
- LEFT JOIN on engagement: a Win user with no Amplitude events appears as a row with NULL engagement columns. That's correct behavior (engagement-zero candidates are included per resolved scope).
- Ordered by `general_election_date DESC` so the freshest rows surface first.

In [ ]:
sdf = spark.sql(f"""
WITH uwc AS (
    SELECT
        u.user_id, u.campaign_id, u.campaign_state, u.election_level, u.office_type,
        u.is_pro, u.icp_office_win, u.viability_score, u.general_election_date,
        c.latest_stage_reached, c.latest_stage_result
    FROM {ANALYTICS}.users_win_candidacy u
    LEFT JOIN {CIVICS}.candidacy c
      ON u.campaign_id = c.product_campaign_id
    WHERE u.is_latest_version AND NOT u.is_demo
      AND u.general_election_date BETWEEN '{WINDOW_START}' AND DATE_SUB('{WINDOW_END}', 1)
      AND c.latest_stage_result IN ('Won', 'Lost')
),
engagement AS (
    SELECT user_id,
           SUM(total_events) AS total_events,
           SUM(activity_days) AS activity_days,
           SUM(campaigns_sent) AS campaigns_sent,
           SUM(dashboard_views) AS dashboard_views
    FROM {DEV}.int__amplitude_win_activity_weekly
    GROUP BY user_id
)
SELECT uwc.*, e.total_events, e.activity_days, e.campaigns_sent, e.dashboard_views
FROM uwc
LEFT JOIN engagement e USING (user_id)
ORDER BY uwc.general_election_date DESC
LIMIT 50
""")

sdf.toPandas()

---

## Wrap-up + verification protocol

To verify a number in INVENTORY.md:
1. Find the section that cites the number (the inventory's "Completeness summary" table lists most of them).
2. Look up the cell in the cross-reference table at the top of this notebook.
3. Run the cell. Compare its output to the value cited.
4. If they disagree, the inventory is stale — update the inventory rather than the notebook.

To re-run the whole inventory under a different scope (different cycle window, different ICP behavior, etc.), change the parameters in the Setup cell and re-execute everything below it.

**Known pending cells:**
- NB-7b (viability calibration on closed races) — pending implementation. Outline: compute observed win rate by `viability_score` band on rows where `latest_stage_result ∈ ('Won', 'Lost')` and the score is non-null.
- NB-8 (path_to_victory schema) — query is in place but may need adjustment based on whether that table contains structured task / recommendation data.
- NB-3 (candidacy_result fallback hazard) — runs against the full civics mart unwindowed; numbers in INVENTORY are aggregate.

In [ ]:
sdf = spark.sql(f"""
with uwc as (
    select user_id, campaign_id, campaign_state, election_level, office_type,
           is_pro, viability_score, general_election_date, general_election_result
    from {ANALYTICS}.users_win_candidacy
    where is_latest_version and not is_demo
      and general_election_result in ('Won', 'Lost')
),
engagement as (
    select user_id,
           sum(total_events) as total_events,
           sum(activity_days) as activity_days,
           sum(campaigns_sent) as campaigns_sent,
           sum(dashboard_views) as dashboard_views
    from {INTERMEDIATE}.int__amplitude_win_activity_weekly
    group by user_id
)
select uwc.*, e.total_events, e.activity_days, e.campaigns_sent, e.dashboard_views
from uwc
left join engagement e using (user_id)
order by uwc.general_election_date desc
limit 50
""")

sdf.toPandas()

---

## Wrap-up

Lift the key numbers from these cells into the `[NB-N]` placeholders in `../INVENTORY.md`. After the lift, walk through the inventory once with real numbers to confirm the narrative still holds, then mark the inventory "verified" in `SESSION_NOTES.md`.